# 40 - Active learning: expanding gold-standard query coverage via uncertainty sampling

Notebook 39's spot-check found the silver-labelling classifier generalises poorly to query topics it never saw during training (only 5 of 101 queries have any gold labels). Rather than randomly picking more candidates to judge, or exhaustively re-judging everything, this uses the classifier's own predicted probability (`silver_prob_relevant`) to find the candidates it is least sure about (probability closest to 0.5) for each of the 101 queries, and builds a small, targeted labeling queue from those.

This also fixes a separate gap: the judge prompt used in notebooks 35/37/39 only showed the judge `name`, `country`, `summary`. The corpus actually has much richer per-company metadata (`organization_type`, `organization_size`, `state`, `district`, `municipality`, `summary_keywords`, `nace_code`) that was never passed to the judge at all. This notebook folds that fix in: the enriched prompt is used for every candidate judged here, including the top-K uncertainty queue.

**Scope**: covers all 101 queries and all candidates already scored in `result/39_silver_labels_full_scale/silver_labels.json` (173,262 rows), not just the 5 pilot queries. K = 10 most-uncertain, never-before-gold-labelled candidates per query, ~1010 candidates total, well within the same cheap cost range already used for the original pooled gold standard.

In [1]:
import json
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("result/40_active_learning_labeling_queue")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

K_PER_QUERY = 10  # most-uncertain never-before-gold candidates to pull per query

silver = pd.read_json("result/39_silver_labels_full_scale/silver_labels.json")
corpus = pd.read_csv("dataset/company_corpus.csv")

# Only candidates that don't already have a real gold label -- no point re-labelling those.
candidates = silver[silver["tier"] == "silver"].copy()
candidates["uncertainty"] = (candidates["silver_prob_relevant"] - 0.5).abs()

print(f"Silver-only candidates eligible for the queue: {len(candidates)}")
print(f"Queries covered: {candidates['query_id'].nunique()}")

Silver-only candidates eligible for the queue: 172845
Queries covered: 101


In [2]:
queue = (
    candidates.sort_values("uncertainty")
    .groupby("query_id", group_keys=False)
    .head(K_PER_QUERY)
    .copy()
)

# company_corpus.csv has exactly one row per domain (each company is tagged to whichever single
# query it originally entered the corpus under), not a full (query, company) matrix. So company
# metadata (name, country, state, ...) must be joined on domain alone, not on (query_id, domain) --
# a candidate considered for query 68 may have entered the corpus under query 12, and its name/
# country/summary/etc. are still correct, they just live under a different query_id in this file.
meta_cols = ["domain", "name", "country", "state", "district", "municipality",
             "organization_type", "organization_size", "summary", "summary_keywords", "nace_code"]
company_meta = corpus[meta_cols].drop_duplicates(subset="domain")
query_text = corpus[["query_id", "query"]].drop_duplicates(subset="query_id")

queue = queue.merge(company_meta, on="domain", how="left")
queue = queue.merge(query_text, on="query_id", how="left")

queue_path = OUTPUT_DIR / "labeling_queue.csv"
queue.to_csv(queue_path, index=False)

print(f"Labeling queue built: {len(queue)} candidates across {queue['query_id'].nunique()} queries")
print(f"Rows still missing company metadata: {queue['name'].isna().sum()} (should be 0)")
print(f"Queries with zero prior gold coverage now included: {sorted(set(queue['query_id']) - set(range(1, 6)))[:10]} ...")
print(f"Saved -> {queue_path}")

Labeling queue built: 1010 candidates across 101 queries
Rows still missing company metadata: 0 (should be 0)
Queries with zero prior gold coverage now included: [6, 7, 8, 9, 10, 11, 12, 13, 14, 15] ...
Saved -> result/40_active_learning_labeling_queue/labeling_queue.csv


## Judge the queue with the enriched prompt

Same multi-judge ensemble pattern as notebook 35 (OpenAI `gpt-4o-mini` + Claude `claude-haiku-4-5`, same models used for the original gold standard, for cost and methodological consistency), same rule: agreement between judges becomes a gold label, disagreement goes to a manual review file rather than being averaged. The only change is the prompt itself, which now includes every metadata field instead of just name/country/summary.

In [3]:
import os, time
import requests
from dotenv import load_dotenv

load_dotenv(override=True)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

ENRICHED_JUDGE_PROMPT_TEMPLATE = """You are judging search result relevance for a company search engine.

Search query: "{query}"

Candidate company:
Name: {name}
Country: {country}
State/region: {state}
District: {district}
Municipality: {municipality}
Organization type: {organization_type}
Organization size: {organization_size}
NACE industry code: {nace_code}
Summary: {summary}
Summary keywords: {summary_keywords}

Rate how relevant this company is to the search query, using exactly one of these labels:
2 = highly relevant (a strong, direct match for the query)
1 = partially relevant (related but not a strong direct match)
0 = not relevant

Respond with ONLY a JSON object: {{"label": <0, 1, or 2>, "reason": "<one short sentence>"}}"""


def build_enriched_prompt(row):
    fields = {c: row.get(c, "") if pd.notna(row.get(c, "")) else "unknown" for c in
              ["query", "name", "country", "state", "district", "municipality",
               "organization_type", "organization_size", "nace_code", "summary", "summary_keywords"]}
    return ENRICHED_JUDGE_PROMPT_TEMPLATE.format(**fields)


def parse_judge_reply(text):
    try:
        start, end = text.index("{"), text.rindex("}") + 1
        parsed = json.loads(text[start:end])
        return int(parsed["label"]), parsed.get("reason", "")
    except (ValueError, KeyError, json.JSONDecodeError):
        return None, f"UNPARSEABLE: {text[:200]}"


def judge_openai(prompt):
    resp = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {OPENAI_API_KEY}", "Content-Type": "application/json"},
        json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}], "temperature": 0},
        timeout=60,
    )
    resp.raise_for_status()
    return parse_judge_reply(resp.json()["choices"][0]["message"]["content"])


def judge_claude(prompt):
    resp = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={"x-api-key": ANTHROPIC_API_KEY, "anthropic-version": "2023-06-01", "Content-Type": "application/json"},
        json={"model": "claude-haiku-4-5-20251001", "max_tokens": 200, "messages": [{"role": "user", "content": prompt}]},
        timeout=60,
    )
    resp.raise_for_status()
    return parse_judge_reply(resp.json()["content"][0]["text"])


JUDGE_FUNCS = {"openai": judge_openai, "claude": judge_claude}
ACTIVE_JUDGES = [name for name, key in [("openai", OPENAI_API_KEY), ("claude", ANTHROPIC_API_KEY)] if key]
print(f"Active judges this run: {ACTIVE_JUDGES}")

Active judges this run: ['openai', 'claude']


In [4]:
CACHE_PATH = OUTPUT_DIR / "judge_cache.json"


def load_cache():
    if CACHE_PATH.exists():
        return json.load(open(CACHE_PATH))
    return {}


def save_cache(cache):
    tmp_path = CACHE_PATH.with_suffix(".json.tmp")
    json.dump(cache, open(tmp_path, "w"), indent=2, default=str)
    tmp_path.replace(CACHE_PATH)


cache = load_cache()
print(f"Loaded {len(cache)} already-judged candidates from cache")

for i, row in queue.iterrows():
    key = f'{row["query_id"]}::{row["domain"]}'
    entry = cache.get(key, {})
    prompt = build_enriched_prompt(row)

    changed = False
    for judge_name in ACTIVE_JUDGES:
        if judge_name in entry:
            continue  # already judged by this one, don't re-spend money on it
        try:
            label, reason = JUDGE_FUNCS[judge_name](prompt)
        except requests.exceptions.RequestException as e:
            print(f"  [{judge_name}] error on {row['domain']}: {e}")
            continue
        entry[judge_name] = {"label": label, "reason": reason}
        changed = True

    if changed:
        cache[key] = entry
        save_cache(cache)  # save after EVERY candidate -- paid calls, never lose progress

    if (i + 1) % 25 == 0 or (i + 1) == len(queue):
        print(f"  {i+1}/{len(queue)} candidates processed")
    time.sleep(0.2)

print("Done for this run (or stopped early -- nothing already judged is lost, rerun to resume).")

Loaded 1010 already-judged candidates from cache
  25/1010 candidates processed
  50/1010 candidates processed
  75/1010 candidates processed
  100/1010 candidates processed
  125/1010 candidates processed
  150/1010 candidates processed
  175/1010 candidates processed
  200/1010 candidates processed
  225/1010 candidates processed
  250/1010 candidates processed
  275/1010 candidates processed
  300/1010 candidates processed
  325/1010 candidates processed
  350/1010 candidates processed
  375/1010 candidates processed
  400/1010 candidates processed
  425/1010 candidates processed
  450/1010 candidates processed
  475/1010 candidates processed
  500/1010 candidates processed
  525/1010 candidates processed
  550/1010 candidates processed
  575/1010 candidates processed
  600/1010 candidates processed
  625/1010 candidates processed
  650/1010 candidates processed
  675/1010 candidates processed
  700/1010 candidates processed
  725/1010 candidates processed
  750/1010 candidates proc

In [5]:
cache = load_cache()
rows = []
for key, entry in cache.items():
    query_id, domain = key.split("::", 1)
    labels = {j: entry[j]["label"] for j in ACTIVE_JUDGES if j in entry and entry[j]["label"] is not None}
    if len(labels) < 2:
        continue  # need at least 2 judges to talk about agreement
    unanimous = len(set(labels.values())) == 1
    rows.append({
        "query_id": int(query_id), "domain": domain, **{f"label_{j}": v for j, v in labels.items()},
        "agree": unanimous, "gold_label": list(labels.values())[0] if unanimous else None,
    })

results_df = pd.DataFrame(rows)
agreed_df = results_df[results_df["agree"]]
disagreed_df = results_df[~results_df["agree"]]

agreed_df.to_json(OUTPUT_DIR / "gold_labels_agreed.json", orient="records", indent=2)
disagreed_df.to_json(OUTPUT_DIR / "needs_manual_review.json", orient="records", indent=2)

print(f"Judged so far: {len(results_df)}")
print(f"Agreed (gold): {len(agreed_df)} ({100*len(agreed_df)/len(results_df):.1f}%)" if len(results_df) else "No results yet")
print(f"Disagreed (needs your review): {len(disagreed_df)}")

Judged so far: 1010
Agreed (gold): 852 (84.4%)
Disagreed (needs your review): 158


In [6]:
cache = load_cache()
queue_lookup = queue.set_index(["query_id", "domain"])

# Preserve any your_label values already filled in on disk -- this cell used to overwrite
# review_queue.csv unconditionally on every rerun, which would silently wipe out completed
# manual labels (discovered after a disconnect forced a full notebook rerun). Never again:
# load whatever's already there first, and carry those labels forward by key.
review_queue_path = OUTPUT_DIR / "review_queue.csv"
existing_labels = {}
if review_queue_path.exists():
    existing = pd.read_csv(review_queue_path)
    for _, r in existing.iterrows():
        val = r.get("your_label", "")
        if pd.notna(val) and str(val).strip() != "":
            existing_labels[(int(r["query_id"]), r["domain"])] = val

review_rows = []
for key, entry in cache.items():
    query_id_str, domain = key.split("::", 1)
    query_id = int(query_id_str)
    labels = {j: entry[j]["label"] for j in ACTIVE_JUDGES if j in entry and entry[j]["label"] is not None}
    if len(labels) < 2 or len(set(labels.values())) == 1:
        continue  # only the disagreements need a human review row

    context = queue_lookup.loc[(query_id, domain)]
    row = {
        "query_id": query_id, "query": context["query"], "domain": domain, "name": context["name"],
        "country": context["country"], "state": context["state"], "nace_code": context["nace_code"],
        "summary": context["summary"],
    }
    for judge_name in ACTIVE_JUDGES:
        if judge_name in entry:
            row[f"{judge_name}_label"] = entry[judge_name]["label"]
            row[f"{judge_name}_reason"] = entry[judge_name]["reason"]
    row["your_label"] = existing_labels.get((query_id, domain), "")  # carry forward if already labelled
    review_rows.append(row)

review_queue_df = pd.DataFrame(review_rows)
review_queue_df.to_csv(review_queue_path, index=False)
n_preserved = sum(1 for r in review_rows if r["your_label"] != "")
print(f"Saved {len(review_queue_df)} disagreements to review -> {review_queue_path}")
print(f"Preserved {n_preserved} already-filled labels from the existing file.")
print("Open this in Excel/Sheets/a text editor, read the query + summary + each judge's label and reason, and fill in the your_label column with 0, 1, or 2.")

Saved 158 disagreements to review -> result/40_active_learning_labeling_queue/review_queue.csv
Preserved 158 already-filled labels from the existing file.
Open this in Excel/Sheets/a text editor, read the query + summary + each judge's label and reason, and fill in the your_label column with 0, 1, or 2.


## Merge into an expanded gold standard

Run this only after `review_queue.csv` has been filled in (`your_label` column filled with 0, 1, 2, or SKIP for every disagreement row). Combines this notebook's new labels with the original 419 from notebook 35 into one expanded gold-labeled dataset, spanning far more of the 101 queries than the original 5-query pilot.

In [7]:
review_final = pd.read_csv(OUTPUT_DIR / "review_queue.csv")
assert (review_final["your_label"].astype(str).str.strip() != "").all(), \
    "Fill in every your_label cell (0, 1, 2, or SKIP) in review_queue.csv before running this cell."

resolved = review_final[review_final["your_label"].astype(str).str.upper() != "SKIP"].copy()
resolved["gold_label"] = resolved["your_label"].astype(int)
resolved["source"] = "human_reviewed_batch2"

agreed_batch2 = agreed_df[["query_id", "domain", "gold_label"]].copy()
agreed_batch2["source"] = "judges_agreed_batch2"

new_gold = pd.concat([agreed_batch2, resolved[["query_id", "domain", "gold_label", "source"]]], ignore_index=True)
new_gold = new_gold.merge(
    queue[["query_id", "domain", "query", "name", "summary"]].drop_duplicates(subset=["query_id", "domain"]),
    on=["query_id", "domain"], how="left",
)

original_gold = pd.read_json("result/35_llm_judge_ensemble/final_gold_labels.json")
expanded_gold = pd.concat([original_gold, new_gold], ignore_index=True)
expanded_gold = expanded_gold.drop_duplicates(subset=["query_id", "domain"], keep="first")

expanded_gold_path = OUTPUT_DIR / "expanded_gold_labels.json"
expanded_gold.to_json(expanded_gold_path, orient="records", indent=2)

print(f"Original gold labels (5 queries): {len(original_gold)}")
print(f"New batch this notebook added   : {len(new_gold)}")
print(f"Expanded gold-labeled dataset   : {len(expanded_gold)} candidates across {expanded_gold['query_id'].nunique()} queries")
print(f"Saved -> {expanded_gold_path}")
print()
print("Label distribution:")
print(expanded_gold["gold_label"].value_counts().sort_index())

Original gold labels (5 queries): 419
New batch this notebook added   : 1010
Expanded gold-labeled dataset   : 1429 candidates across 101 queries
Saved -> result/40_active_learning_labeling_queue/expanded_gold_labels.json

Label distribution:
gold_label
0.0     63
1.0    381
2.0    985
Name: count, dtype: int64
